<a href="https://colab.research.google.com/github/candle16/203-accelerate/blob/master/Crime_Analysis_Canada.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:

# Configure Git with your info
!git config --global user.name "candle16"
!git config --global user.email "kateskipton@gmail.com"

# Store credentials so you don't have to enter token every time
!git config --global credential.helper store.

In [23]:
# Clone your repository
!git clone https://github.com/candle16/statcan-crime-analysis.git

# Navigate into the folder
%cd statcan-crime-analysis

# Check that you're in the right place
!pwd

Cloning into 'statcan-crime-analysis'...
remote: Enumerating objects: 4, done.
remote: Counting objects: 100% (4/4), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 4 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (4/4), done.
/content/statcan-crime-analysis/statcan-crime-analysis/statcan-crime-analysis
/content/statcan-crime-analysis/statcan-crime-analysis/statcan-crime-analysis


In [24]:
!mkdir -p data
!mkdir -p outputs
!mkdir -p scripts

In [25]:
print('Hello world')

Hello world


In [26]:
# CELL 1: Install Dependencies
!pip install stats-can openpyxl

In [27]:
%%writefile scripts/analysis.py

# CELL 2: Import Libraries
import pandas as pd
import stats_can as sc
import logging
from pathlib import Path
from IPython.display import display

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(message)s'
)
logger = logging.getLogger(__name__)

Writing scripts/analysis.py


In [28]:
# CELL 3: RetailCrimeAnalyzer Class Definition
class RetailCrimeAnalyzer:
    """Generates complete analysis tables from live StatCan data."""
    def __init__(self):
        self.data = {}
        self.tables = {}
        self.calculations = {}

    def fetch_data(self):
        """Fetch all required data from Statistics Canada."""
        logger.info("=" * 80)
        logger.info("FETCHING DATA FROM STATISTICS CANADA")
        logger.info("=" * 80)
        logger.info("This will take 2-3 minutes on first run...")
        logger.info("(Data is cached locally for instant access next time)")
        # Fetch incident-based crime data
        logger.info("Downloading Table 35-10-0177-01 (Incident-based crime)...")
        try:
            df_incidents = sc.zip_table_to_dataframe("35100177")
            logger.info(f"Downloaded {len(df_incidents):,} rows")
            # IMMEDIATELY filter to reduce memory
            logger.info("Filtering for Canada only...")
            df_incidents = df_incidents[df_incidents['GEO'] == 'Canada'].copy()
            logger.info(f"Reduced to {len(df_incidents):,} rows")
            logger.debug(f"Incident data columns: {df_incidents.columns.tolist()}")
            self.data['incidents'] = df_incidents
        except Exception as e:
            logger.error(f"Error fetching incidents data: {str(e)}", exc_info=True)
            return False
        # Fetch organized crime data
        logger.info("Downloading Table 35-10-0062-01 (Organized crime)...")
        try:
            df_organized = sc.zip_table_to_dataframe("35100062")
            logger.info(f"Downloaded {len(df_organized):,} rows")
            # Filter immediately
            logger.info("Filtering for Canada only...")
            df_organized = df_organized[df_organized['GEO'].str.contains('Canada', case=False)].copy()
            logger.info(f"Reduced to {len(df_organized):,} rows")
            logger.debug(f"Organized data columns: {df_organized.columns.tolist()}")
            self.data['organized'] = df_organized
        except Exception as e:
            logger.error(f"Error fetching organized data: {str(e)}", exc_info=True)
            return False
        logger.info("All data fetched and filtered!")
        return True

    def extract_shoplifting_data(self):
        """Extract shoplifting data for 2016-2024."""
        logger.info("Extracting shoplifting data (2016-2024)...")
        try:
            df = self.data['incidents']
            logger.debug(f"REF_DATE sample: {df['REF_DATE'].head()}")
            logger.debug(f"REF_DATE dtype: {df['REF_DATE'].dtype}")
            # Pre-filter the dataframe for performance
            logger.info("Filtering data...")
            df_filtered = df[df['REF_DATE'].dt.year.between(2016, 2024)].copy()
            logger.info(f"Working with {len(df_filtered):,} rows")
            logger.debug(f"Unique Violations sample: {df_filtered['Violations'].unique()[:10]}")
            logger.debug(f"Unique Statistics: {df_filtered['Statistics'].unique()}")
            results = {
                'under_5k': {},
                'over_5k': {}
            }
            logger.info("Processing years: ")
            for year in range(2016, 2025):
                logger.debug(f"Processing year {year}...")
                # Shoplifting under $5K - incidents
                mask = (
                        df_filtered['Violations'].str.contains('shoplifting', case=False, na=False) &
                        df_filtered['Violations'].str.contains('5,000 or under', case=False, na=False) &
                        (df_filtered['Statistics'] == 'Actual incidents') &
                        (df_filtered['REF_DATE'].dt.year == year)
                )
                incidents_data = df_filtered[mask]
                if len(incidents_data) > 0:
                    incidents = int(incidents_data['VALUE'].iloc[0])
                    # Get rate
                    rate_mask = (
                            df_filtered['Violations'].str.contains('shoplifting', case=False, na=False) &
                            df_filtered['Violations'].str.contains('5,000 or under', case=False, na=False) &
                            (df_filtered['Statistics'] == 'Rate per 100,000 population') &
                            (df_filtered['REF_DATE'].dt.year == year)
                    )
                    rate_data = df_filtered[rate_mask]
                    rate = float(rate_data['VALUE'].iloc[0]) if len(rate_data) > 0 else 0.0
                    results['under_5k'][year] = {
                        'incidents': incidents,
                        'rate': rate
                    }
                    logger.debug(f"Found under_5k for {year}: {incidents}")
                else:
                    logger.warning(f"No data for Shoplifting Under $5K in {year}")
                # Shoplifting over $5K - incidents
                mask = (
                        df_filtered['Violations'].str.contains('shoplifting', case=False, na=False) &
                        df_filtered['Violations'].str.contains('over.*5,000', case=False, na=False) &
                        (df_filtered['Statistics'] == 'Actual incidents') &
                        (df_filtered['REF_DATE'].dt.year == year)
                )
                incidents_data = df_filtered[mask]
                if len(incidents_data) > 0:
                    incidents = int(incidents_data['VALUE'].iloc[0])
                    # Get rate
                    rate_mask = (
                            df_filtered['Violations'].str.contains('shoplifting', case=False, na=False) &
                            df_filtered['Violations'].str.contains('over.*5,000', case=False, na=False) &
                            (df_filtered['Statistics'] == 'Rate per 100,000 population') &
                            (df_filtered['REF_DATE'].dt.year == year)
                    )
                    rate_data = df_filtered[rate_mask]
                    rate = float(rate_data['VALUE'].iloc[0]) if len(rate_data) > 0 else 0.0
                    results['over_5k'][year] = {
                        'incidents': incidents,
                        'rate': rate
                    }
                    logger.debug(f"Found over_5k for {year}: {incidents}")
                else:
                    logger.warning(f"No data for Shoplifting Over $5K in {year}")
            self.data['shoplifting'] = results
            logger.info("Shoplifting data extracted")
            return results
        except Exception as e:
            logger.error(f"Error in extract_shoplifting_data: {str(e)}", exc_info=True)
            return None

    def extract_organized_crime_data(self):
        """Extract organized crime data for 2016-2024."""
        logger.info("Extracting organized crime data (2016-2024)...")
        try:
            df = self.data['organized']
            logger.debug(f"REF_DATE sample: {df['REF_DATE'].head()}")
            logger.debug(f"REF_DATE dtype: {df['REF_DATE'].dtype}")
            # Pre-filter
            df_filtered = df[df['REF_DATE'].dt.year.between(2016, 2024)].copy()
            logger.info(f"Filtered organized data to {len(df_filtered):,} rows for years 2016-2024")
            logger.debug(f"Unique Most serious violation sample: {df_filtered['Most serious violation'].unique()[:10]}")
            results = {
                'total': {},
                'under_5k': {},
                'over_5k': {}
            }
            logger.info("Processing years: ")
            for year in range(2016, 2025):
                logger.debug(f"Processing year {year}...")
                # Total organized crime
                mask = (
                        (df_filtered['Most serious violation'] == 'Total, all violations') &
                        (df_filtered['REF_DATE'].dt.year == year)
                )
                data = df_filtered[mask]
                if len(data) > 0:
                    results['total'][year] = int(data['VALUE'].iloc[0])
                    logger.debug(f"Found total for {year}: {results['total'][year]}")
                else:
                    logger.warning(f"No data for Total organized crime in {year}")
                # Organized shoplifting under $5K
                mask = (
                        df_filtered['Most serious violation'].str.contains('shoplifting', case=False, na=False) &
                        df_filtered['Most serious violation'].str.contains('5,000 or under', case=False, na=False) &
                        (df_filtered['REF_DATE'].dt.year == year)
                )
                data = df_filtered[mask]
                if len(data) > 0:
                    results['under_5k'][year] = int(data['VALUE'].iloc[0])
                    logger.debug(f"Found under_5k for {year}: {results['under_5k'][year]}")
                else:
                    logger.warning(f"No data for Organized shoplifting under $5K in {year}")
                # Organized shoplifting over $5K
                mask = (
                        df_filtered['Most serious violation'].str.contains('shoplifting', case=False, na=False) &
                        df_filtered['Most serious violation'].str.contains('over.*5,000', case=False, na=False) &
                        (df_filtered['REF_DATE'].dt.year == year)
                )
                data = df_filtered[mask]
                if len(data) > 0:
                    results['over_5k'][year] = int(data['VALUE'].iloc[0])
                    logger.debug(f"Found over_5k for {year}: {results['over_5k'][year]}")
                else:
                    logger.warning(f"No data for Organized shoplifting over $5K in {year}")
            self.data['organized_crime'] = results
            logger.info("Organized crime data extracted")
            return results
        except Exception as e:
            logger.error(f"Error in extract_organized_crime_data: {str(e)}", exc_info=True)
            return None

    def calculate_growth_metrics(self):
        """Calculate all growth percentages and multipliers."""
        logger.info("Calculating growth metrics...")
        try:
            shop_data = self.data['shoplifting']
            org_data = self.data['organized_crime']
            baseline_year = 2019
            current_year = 2024
            calcs = {}
            # Shoplifting under $5K
            if baseline_year in shop_data['under_5k'] and current_year in shop_data['under_5k']:
                baseline = shop_data['under_5k'][baseline_year]['incidents']
                current = shop_data['under_5k'][current_year]['incidents']
                calcs['shop_under_5k'] = {
                    'baseline': baseline,
                    'current': current,
                    'growth_pct': round(((current - baseline) / baseline) * 100, 2),
                    'multiplier': round(current / baseline, 2)
                }
                logger.debug(f"Calculated shop_under_5k: {calcs['shop_under_5k']}")
            # Shoplifting over $5K
            if baseline_year in shop_data['over_5k'] and current_year in shop_data['over_5k']:
                baseline = shop_data['over_5k'][baseline_year]['incidents']
                current = shop_data['over_5k'][current_year]['incidents']
                calcs['shop_over_5k'] = {
                    'baseline': baseline,
                    'current': current,
                    'growth_pct': round(((current - baseline) / baseline) * 100, 2),
                    'multiplier': round(current / baseline, 2)
                }
                logger.debug(f"Calculated shop_over_5k: {calcs['shop_over_5k']}")
            # New: Total Shoplifting (sum of under + over)
            if baseline_year in shop_data['under_5k'] and baseline_year in shop_data['over_5k'] and current_year in shop_data['under_5k'] and current_year in shop_data['over_5k']:
                baseline = shop_data['under_5k'][baseline_year]['incidents'] + shop_data['over_5k'][baseline_year]['incidents']
                current = shop_data['under_5k'][current_year]['incidents'] + shop_data['over_5k'][current_year]['incidents']
                calcs['shop_total'] = {
                    'baseline': baseline,
                    'current': current,
                    'growth_pct': round(((current - baseline) / baseline) * 100, 2),
                    'multiplier': round(current / baseline, 2)
                }
                logger.debug(f"Calculated shop_total: {calcs['shop_total']}")
            # Total organized crime
            if baseline_year in org_data['total'] and current_year in org_data['total']:
                baseline = org_data['total'][baseline_year]
                current = org_data['total'][current_year]
                calcs['org_total'] = {
                    'baseline': baseline,
                    'current': current,
                    'growth_pct': round(((current - baseline) / baseline) * 100, 2),
                    'multiplier': round(current / baseline, 2)
                }
                logger.debug(f"Calculated org_total: {calcs['org_total']}")
            # Organized shoplifting under $5K
            if baseline_year in org_data['under_5k'] and current_year in org_data['under_5k']:
                baseline = org_data['under_5k'][baseline_year]
                current = org_data['under_5k'][current_year]
                calcs['org_under_5k'] = {
                    'baseline': baseline,
                    'current': current,
                    'growth_pct': round(((current - baseline) / baseline) * 100, 2),
                    'multiplier': round(current / baseline, 2)
                }
                logger.debug(f"Calculated org_under_5k: {calcs['org_under_5k']}")
            # Organized shoplifting over $5K
            if baseline_year in org_data['over_5k'] and current_year in org_data['over_5k']:
                baseline = org_data['over_5k'][baseline_year]
                current = org_data['over_5k'][current_year]
                calcs['org_over_5k'] = {
                    'baseline': baseline,
                    'current': current,
                    'growth_pct': round(((current - baseline) / baseline) * 100, 2),
                    'multiplier': round(current / baseline, 2)
                }
                logger.debug(f"Calculated org_over_5k: {calcs['org_over_5k']}")
            # Total Org. Shoplifting (sum of under + over)
            if baseline_year in org_data['under_5k'] and baseline_year in org_data['over_5k'] and current_year in org_data['under_5k'] and current_year in org_data['over_5k']:
                baseline = org_data['under_5k'][baseline_year] + org_data['over_5k'][baseline_year]
                current = org_data['under_5k'][current_year] + org_data['over_5k'][current_year]
                calcs['org_total_shoplifting'] = {
                    'baseline': baseline,
                    'current': current,
                    'growth_pct': round(((current - baseline) / baseline) * 100, 2),
                    'multiplier': round(current / baseline, 2)
                }
                logger.debug(f"Calculated org_total_shoplifting: {calcs['org_total_shoplifting']}")
            self.calculations = calcs
            logger.info("All metrics calculated")
            return calcs
        except Exception as e:
            logger.error(f"Error in calculate_growth_metrics: {str(e)}", exc_info=True)
            return None

    def generate_table_1_incident_based(self):
        """Generate Table 1: All Incident-Based Crime Statistics."""
        logger.info("Generating Table 1: Incident-Based Crime (2016-2024)...")
        try:
            shop_data = self.data['shoplifting']
            rows = []
            years = range(2016, 2025)
            for year in years:
                if year in shop_data['under_5k']:
                    data = shop_data['under_5k'][year]
                    rows.append({
                        'Category': 'Shoplifting Under $5,000',
                        'Year': year,
                        'Incidents': data['incidents'],
                        'Rate per 100,000': round(data['rate'], 2)
                    })
            for year in years:
                if year in shop_data['over_5k']:
                    data = shop_data['over_5k'][year]
                    rows.append({
                        'Category': 'Shoplifting Over $5,000',
                        'Year': year,
                        'Incidents': data['incidents'],
                        'Rate per 100,000': round(data['rate'], 2)
                    })
            df = pd.DataFrame(rows)
            self.tables['incident_based'] = df
            logger.info("Table 1 created")
            return df
        except Exception as e:
            logger.error(f"Error in generate_table_1_incident_based: {str(e)}", exc_info=True)
            return None

    def generate_table_2_organized_crime(self):
        """Generate Table 2: Organized Crime Statistics."""
        logger.info("Generating Table 2: Organized Crime (2016-2024)...")
        try:
            org_data = self.data['organized_crime']
            rows = []
            years = range(2016, 2025)
            for year in years:
                if year in org_data['total']:
                    rows.append({
                        'Category': 'Total Organized Crime',
                        'Year': year,
                        'Incidents': org_data['total'][year]
                    })
            for year in years:
                if year in org_data['under_5k']:
                    rows.append({
                        'Category': 'Organized Shoplifting Under $5,000',
                        'Year': year,
                        'Incidents': org_data['under_5k'][year]
                    })
            for year in years:
                if year in org_data['over_5k']:
                    rows.append({
                        'Category': 'Organized Shoplifting Over $5,000',
                        'Year': year,
                        'Incidents': org_data['over_5k'][year]
                    })
            # Add Total Org. Shoplifting rows
            for year in years:
                if year in org_data['under_5k'] and year in org_data['over_5k']:
                    incidents = org_data['under_5k'][year] + org_data['over_5k'][year]
                    rows.append({
                        'Category': 'Total Organized Shoplifting',
                        'Year': year,
                        'Incidents': incidents
                    })
            df = pd.DataFrame(rows)
            self.tables['organized_crime'] = df
            logger.info("Table 2 created")
            return df
        except Exception as e:
            logger.error(f"Error in generate_table_2_organized_crime: {str(e)}", exc_info=True)
            return None

    def generate_table_3_growth_summary(self):
        """Generate Table 3: Growth Summary (2019 vs 2024)."""
        logger.info("Generating Table 3: Growth Summary...")
        try:
            calcs = self.calculations
            rows = []
            # Define the order and names
            ordered_keys = [
                ('shop_under_5k', 'Shoplifting Under $5,000'),
                ('shop_over_5k', 'Shoplifting Over $5,000'),
                ('shop_total', 'Total Shoplifting'),
                ('org_total', 'Total Organized Crime'),
                ('org_under_5k', 'Organized Shoplifting Under $5,000'),
                ('org_over_5k', 'Organized Shoplifting Over $5,000'),
                ('org_total_shoplifting', 'Total Organized Shoplifting')
            ]
            for key, name in ordered_keys:
                if key in calcs:
                    data = calcs[key]
                    rows.append({
                        'Category': name,
                        '2019 Baseline': data['baseline'],
                        '2024 Current': data['current'],
                        'Growth %': data['growth_pct'],
                        'Multiplier': data['multiplier']
                    })
            df = pd.DataFrame(rows)
            self.tables['growth_summary'] = df
            logger.info("Table 3 created")
            return df
        except Exception as e:
            logger.error(f"Error in generate_table_3_growth_summary: {str(e)}", exc_info=True)
            return None

    def save_to_excel(self, filename='retail_crime_analysis.xlsx'):
        """Save all tables to a formatted Excel file."""
        logger.info(f"Saving to Excel: {filename}...")
        try:
            with pd.ExcelWriter(filename, engine='openpyxl') as writer:
                if 'incident_based' in self.tables:
                    self.tables['incident_based'].to_excel(writer, sheet_name='Incident Based Crime', index=False)
                if 'organized_crime' in self.tables:
                    self.tables['organized_crime'].to_excel(writer, sheet_name='Organized Crime', index=False)
                if 'growth_summary' in self.tables:
                    self.tables['growth_summary'].to_excel(writer, sheet_name='Growth Summary', index=False)
            logger.info("Excel file saved")
        except Exception as e:
            logger.error(f"Error saving to Excel: {str(e)}", exc_info=True)

    def save_to_csv(self, output_dir='output'):
        """Save each table as a separate CSV file."""
        logger.info(f"Saving CSV files to {output_dir}/...")
        try:
            Path(output_dir).mkdir(exist_ok=True)
            for name, df in self.tables.items():
                filename = f"{output_dir}/{name}.csv"
                df.to_csv(filename, index=False)
                logger.info(f"Saved {filename}")
        except Exception as e:
            logger.error(f"Error saving to CSV: {str(e)}", exc_info=True)

    def print_summary(self):
        """Print a summary of key findings."""
        logger.info("=" * 80)
        logger.info("ANALYSIS SUMMARY")
        logger.info("=" * 80)
        calcs = self.calculations
        logger.info("Key Growth Metrics (2019 → 2024):")
        if 'shop_under_5k' in calcs:
            data = calcs['shop_under_5k']
            logger.info(f"Shoplifting Under $5K:")
            logger.info(f"  {data['baseline']:,} → {data['current']:,}")
            logger.info(f"  Growth: +{data['growth_pct']}% ({data['multiplier']}x)")
        if 'shop_over_5k' in calcs:
            data = calcs['shop_over_5k']
            logger.info(f"Shoplifting Over $5K:")
            logger.info(f"  {data['baseline']:,} → {data['current']:,}")
            logger.info(f"  Growth: +{data['growth_pct']}% ({data['multiplier']}x)")
        if 'shop_total' in calcs:
            data = calcs['shop_total']
            logger.info(f"Total Shoplifting:")
            logger.info(f"  {data['baseline']:,} → {data['current']:,}")
            logger.info(f"  Growth: +{data['growth_pct']}% ({data['multiplier']}x)")
        if 'org_total' in calcs:
            data = calcs['org_total']
            logger.info(f"Total Organized Crime:")
            logger.info(f"  {data['baseline']:,} → {data['current']:,}")
            logger.info(f"  Growth: +{data['growth_pct']}% ({data['multiplier']}x)")
        if 'org_under_5k' in calcs:
            data = calcs['org_under_5k']
            logger.info(f"Organized Shoplifting Under $5K:")
            logger.info(f"  {data['baseline']:,} → {data['current']:,}")
            logger.info(f"  Growth: +{data['growth_pct']}% ({data['multiplier']}x)")
        if 'org_over_5k' in calcs:
            data = calcs['org_over_5k']
            logger.info(f"Organized Shoplifting Over $5K:")
            logger.info(f"  {data['baseline']} → {data['current']}")
            logger.info(f"  Growth: +{data['growth_pct']}% ({data['multiplier']}x) ⚠️ Small base")
        if 'org_total_shoplifting' in calcs:
            data = calcs['org_total_shoplifting']
            logger.info(f"Total Organized Shoplifting:")
            logger.info(f"  {data['baseline']:,} → {data['current']:,}")
            logger.info(f"  Growth: +{data['growth_pct']}% ({data['multiplier']}x)")
        logger.info("=" * 80)

    def display_tables(self):
        """Display tables in Colab cell outputs for easy diagnosis."""
        logger.info("Displaying tables for diagnosis:")
        if 'incident_based' in self.tables:
            logger.info("\nIncident Based Crime Table:")
            display(self.tables['incident_based'])
        if 'organized_crime' in self.tables:
            logger.info("\nOrganized Crime Table:")
            display(self.tables['organized_crime'])
        if 'growth_summary' in self.tables:
            logger.info("\nGrowth Summary Table:")
            display(self.tables['growth_summary'])

In [29]:
# CELL 4: Run the Analysis
# Create analyzer instance
analyzer = RetailCrimeAnalyzer()

# Fetch data
if analyzer.fetch_data():
    # Extract data
    analyzer.extract_shoplifting_data()
    analyzer.extract_organized_crime_data()

    # Calculate metrics
    analyzer.calculate_growth_metrics()

    # Generate tables
    analyzer.generate_table_1_incident_based()
    analyzer.generate_table_2_organized_crime()
    analyzer.generate_table_3_growth_summary()

    # Display results
    analyzer.print_summary()
    analyzer.display_tables()

    # Save outputs
    analyzer.save_to_excel('retail_crime_analysis.xlsx')
    analyzer.save_to_csv('output')

    print("\n✅ Analysis complete!")
    print("📊 Files saved:")
    print("  - retail_crime_analysis.xlsx")
    print("  - output/incident_based.csv")
    print("  - output/organized_crime.csv")
    print("  - output/growth_summary.csv")
else:
    print("❌ Failed to fetch data")

35100177-eng.zip: 100%|██████████| 100M/100M [00:30<00:00, 3.31MB/s] 
/usr/local/lib/python3.12/dist-packages/stats_can/sc.py:221: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  df["REF_DATE"] = pd.to_datetime(df["REF_DATE"], errors="ignore")
35100062-eng.zip: 100%|██████████| 10.1k/10.1k [00:00<00:00, 2.60MB/s]
/usr/local/lib/python3.12/dist-packages/stats_can/sc.py:221: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  df["REF_DATE"] = pd.to_datetime(df["REF_DATE"], errors="ignore")


,Category,Year,Incidents,"Rate per 100,000"
0,"Shoplifting Under $5,000",2016,102948,285.09
1,"Shoplifting Under $5,000",2017,108313,296.38
2,"Shoplifting Under $5,000",2018,124896,336.90
3,"Shoplifting Under $5,000",2019,140415,373.26
4,"Shoplifting Under $5,000",2020,91347,240.21
5,"Shoplifting Under $5,000",2021,95242,249.06
6,"Shoplifting Under $5,000",2022,127940,328.59
7,"Shoplifting Under $5,000",2023,155792,388.67
8,"Shoplifting Under $5,000",2024,182361,441.67
9,"Shoplifting Over $5,000",2016,574,1.59


,Category,Year,Incidents
0,Total Organized Crime,2016,4810
1,Total Organized Crime,2017,6184
2,Total Organized Crime,2018,6436
3,Total Organized Crime,2019,8519
4,Total Organized Crime,2020,10970
5,Total Organized Crime,2021,10307
6,Total Organized Crime,2022,11170
7,Total Organized Crime,2023,13167
8,Total Organized Crime,2024,14804
9,"Organized Shoplifting Under $5,000",2016,62


,Category,2019 Baseline,2024 Current,Growth %,Multiplier
0,"Shoplifting Under $5,000",140415,182361,29.87,1.30
1,"Shoplifting Over $5,000",703,1666,136.98,2.37
2,Total Shoplifting,141118,184027,30.41,1.30
3,Total Organized Crime,8519,14804,73.78,1.74
4,"Organized Shoplifting Under $5,000",115,220,91.30,1.91
5,"Organized Shoplifting Over $5,000",7,48,585.71,6.86
6,Total Organized Shoplifting,122,268,119.67,2.20



✅ Analysis complete!
📊 Files saved:
  - retail_crime_analysis.xlsx
  - output/incident_based.csv
  - output/organized_crime.csv
  - output/growth_summary.csv


In [30]:
# CELL 5 (OPTIONAL): Download files to your computer
from google.colab import files

# Download Excel file
files.download('retail_crime_analysis.xlsx')

# Download CSV files
files.download('output/incident_based.csv')
files.download('output/organized_crime.csv')
files.download('output/growth_summary.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [31]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	35100062-eng.zip
	35100062.json
	35100177-eng.zip
	35100177.json
	output/
	retail_crime_analysis.xlsx
	scripts/

nothing added to commit but untracked files present (use "git add" to track)


In [32]:
!git add

Nothing specified, nothing added.
hint: Maybe you wanted to say 'git add .'?
hint: Turn this message off by running
hint: "git config advice.addEmptyPathspec false"


In [33]:
!git commit -m "Test git upload with statcan analysis code"

On branch main
Your branch is up to date with 'origin/main'.

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	35100062-eng.zip
	35100062.json
	35100177-eng.zip
	35100177.json
	output/
	retail_crime_analysis.xlsx
	scripts/

nothing added to commit but untracked files present (use "git add" to track)
